In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import src.imports as lib

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# ĐỌC DỮ LIỆU ĐÃ GHÉP
# ==========================================

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import src.imports as lib

file_path = lib.Path("../data/data-clean/hoc_sinh_lop6_9_clean.xlsx")

df_merged = lib.pd.read_excel(
    file_path,
    engine="openpyxl"
)

print(f"Kích thước dữ liệu: {df_merged.shape}")
display(df_merged.head())

In [ ]:
# ==========================================
# 03 - FEATURE & TARGET SEPARATION
# XÁC ĐỊNH 96 ĐẶC TRƯNG ĐIỂM SỐ LÀM TẬP X
# ==========================================

# 1. Xác định các cột điểm lớp 6–9
score_cols = [
    col for col in df_merged.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

# 2. Tạo tập đặc trưng X
X = df_merged[score_cols].copy()

# 3. Kiểm tra cấu trúc
assert len(score_cols) == 96, "Số đặc trưng phải bằng 96!"
assert X.shape == (10000, 96), "Kích thước tập X không hợp lệ!"
assert X.columns.is_unique, "Phát hiện tên đặc trưng trùng lặp!"

print("=" * 60)
print("TẬP ĐẶC TRƯNG X")
print("=" * 60)

print(f"Số học sinh: {X.shape[0]:,}")
print(f"Số đặc trưng: {X.shape[1]}")
print(f"Kích thước X: {X.shape}")

for grade in [6, 7, 8, 9]:
    count = sum(col.startswith(f"L{grade}_") for col in X.columns)
    print(f"Lớp {grade}: {count} đặc trưng")

display(X.head())

In [ ]:
# ==========================================
# 03 - FEATURE & TARGET SEPARATION
# XÁC ĐỊNH X, y VÀ KIỂM TRA DỮ LIỆU ĐẦU VÀO
# ==========================================

# 1. Xác định 9 nhãn mục tiêu
choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

y = df_merged[choice_cols].copy()

# 2. Tách mã học sinh để quản lý dữ liệu
student_ids = df_merged["ma_hoc_sinh"].copy()

# 3. Xác định lại tập X gồm 96 đặc trưng điểm số
score_cols = [
    col for col in df_merged.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

X = df_merged[score_cols].copy()

# 4. Kiểm tra không có nhãn mục tiêu hoặc ID trong X
target_leakage = set(X.columns) & set(choice_cols)
id_leakage = "ma_hoc_sinh" in X.columns

assert not target_leakage, f"Phát hiện nhãn trong X: {target_leakage}"
assert not id_leakage, "Phát hiện ma_hoc_sinh trong X!"

# 5. Kiểm tra kiểu dữ liệu đầu vào
X_numeric = all(lib.pd.api.types.is_numeric_dtype(dtype) for dtype in X.dtypes)
y_numeric = all(lib.pd.api.types.is_numeric_dtype(dtype) for dtype in y.dtypes)
y_binary = y.isin([0, 1]).all().all()

assert X_numeric, "Tập X chứa cột không phải kiểu số!"
assert y_numeric, "Tập y chứa cột không phải kiểu số!"
assert y_binary, "Tập y chứa giá trị ngoài {0, 1}!"

# 6. Xác nhận hình dạng dữ liệu
assert X.shape == (10000, 96), f"Kích thước X không hợp lệ: {X.shape}"
assert y.shape == (10000, 9), f"Kích thước y không hợp lệ: {y.shape}"
assert len(student_ids) == 10000, "Số mã học sinh không hợp lệ!"
assert student_ids.is_unique, "Mã học sinh bị trùng!"
assert X.index.equals(y.index), "Chỉ số X và y không khớp!"

# 7. Tổng hợp kết quả
validation_report = lib.pd.DataFrame({
    "Tiêu chí": [
        "Số học sinh",
        "Số đặc trưng X",
        "Số nhãn y",
        "Nhãn mục tiêu xuất hiện trong X",
        "Mã học sinh xuất hiện trong X",
        "Kiểu dữ liệu X",
        "Kiểu dữ liệu y",
        "Giá trị nhãn 0/1",
        "Hình dạng X",
        "Hình dạng y"
    ],
    "Kết quả": [
        len(student_ids),
        X.shape[1],
        y.shape[1],
        len(target_leakage),
        id_leakage,
        "Numeric" if X_numeric else "Non-numeric",
        "Numeric" if y_numeric else "Non-numeric",
        "Hợp lệ" if y_binary else "Không hợp lệ",
        str(X.shape),
        str(y.shape)
    ]
})

print("=" * 60)
print("KIỂM TRA FEATURE & TARGET SEPARATION")
print("=" * 60)

display(validation_report)

print("\nPASS: Tách dữ liệu X, y và ma_hoc_sinh thành công.")

print("\nTẬP ĐẶC TRƯNG X:")
display(X.head())

print("\nTẬP NHÃN y:")
display(y.head())